# generator-loss-fool-discriminator — worked example 1: Non-saturating generator loss with BCE

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-loss-fool-discriminator`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The non-saturating generator loss is `-log(D(G(z)))`, implemented as binary cross-entropy between D's probability on the fakes and an all-ones target. The generator labels its own fakes as 'real' (target 1) so the gradient pushes D's verdict toward 1 — the opposite label the discriminator uses for the same fakes.

## Worked solution

We compute the generator's loss given D's probabilities on the current fake batch.

1. `d_pred_fake` is D's sigmoid output on `G(z)`, shape `(B,)`, each in `(0, 1)`.
2. The generator *wants* D to call these real, so the target is all ones: `targets = t.ones_like(d_pred_fake)`. Using `ones_like` guarantees matching shape, dtype, and device.
3. `F.binary_cross_entropy(d_pred_fake, targets)` evaluates the mean of `-log(d_pred_fake)` — exactly the non-saturating loss. It is small when D is fooled (prediction near 1) and large when D is confident the fakes are fake (prediction near 0).
4. We print the loss at a fooled batch and a not-fooled batch to show the monotone decrease in D's confidence.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(0)

def generator_loss(d_pred_fake):
    targets = t.ones_like(d_pred_fake)
    return F.binary_cross_entropy(d_pred_fake, targets)

fooled = t.full((8,), 0.9)
not_fooled = t.full((8,), 0.1)
print('fooled loss:', round(float(generator_loss(fooled)), 4))
print('not-fooled loss:', round(float(generator_loss(not_fooled)), 4))
print('fooled < not_fooled:', bool(generator_loss(fooled) < generator_loss(not_fooled)))